# Working with CSV and JSON: Options, Modes, and Nested Structures

**Module**: 08_data_sources_and_formats  
**Notebook**: 01_csv_and_json.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Reading and writing CSV and JSON files, handling read modes (PERMISSIVE, DROPMALFORMED, FAILFAST), and flattening nested JSON payloads.
- **Why does this topic exist?**: CSV and JSON are ubiquitous raw ingestion formats in modern data platforms.
- **What problem does it solve?**: Corrupted rows crashing pipelines or silent type mismatches.
- **Where is it used in real-world Data Engineering?**: Raw bronze-layer ingestion pipelines in data lakes.

---

## 2. Read Modes: PERMISSIVE (default) vs DROPMALFORMED vs FAILFAST

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("CSV_JSON_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Read CSV with options
sales = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/sales.csv")
sales.show(3)

# Read nested JSON from retail/order_events.json
events = spark.read.json("/opt/spark-data/retail/order_events.json")
events.printSchema()
events.select("event_id", "order.order_id", "customer.country").show(3)

---

## Experiment: Testing FAILFAST on Corrupt CSV

- **1. What are we testing?**: Attempt to read dirty CSV in FAILFAST mode.
- **2. What do we expect?**: Spark throws an exception immediately on first bad row.

In [ ]:
try:
    spark.read.option("header", True).option("mode", "FAILFAST").schema("order_id int, date date").csv("/opt/spark-data/quality/orders_dirty.csv").count()
except Exception as e:
    print(f"FAILFAST successfully threw error on malformed data: {type(e).__name__}")

**3. What happened?**: Threw exception upon parsing invalid dates/ints.  
**4. Why did it happen?**: FAILFAST stops execution immediately.  
**5. Practical Engineering Lesson**: Use PERMISSIVE with columnNameOfCorruptRecord for resilient batch ingestion.

---

## Practical Exercises

### Exercise 1 (Beginner): Read Multi-line JSON
**Problem Statement**: Set multiline option to true.  
**Input / Context**: json

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Flatten Struct
**Problem Statement**: Flatten nested customer struct.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Write JSON
**Problem Statement**: Write sales to JSON format.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Read Multi-line JSON):
```python
print('spark.read.option("multiline", True).json(path)')
```

#### Solution 2 (Flatten Struct):
```python
events.select('customer.*').show(3)
```

#### Solution 3 (Write JSON):
```python
sales.write.mode('overwrite').json('/opt/spark-data/warehouse/sales_json')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Modes**: PERMISSIVE tracks bad rows; FAILFAST throws; DROPMALFORMED discards.
- **JSON**: Natively parsed into StructType.

### Essential Technical Terms
- `PERMISSIVE`: Default parsing mode.
- `columnNameOfCorruptRecord`: Captures raw bad string.

### Common Engineering Mistakes
- **Using text formats for petabyte analytics**: CSV/JSON lack columnar compression and predicate pushdown.

### Interview Questions on This Topic
- **Q: What are the three read modes in Spark?**
  - *A*: PERMISSIVE, DROPMALFORMED, and FAILFAST.

### What to Study Next
Open 02_parquet_and_orc.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')